In [0]:
%python
customers_df = spark.read.table("customers")
display(customers_df)

In [0]:
display(customers_df.select("customer_id","email"))

In [0]:
from pyspark.sql import functions as F
display(customers_df.select(F.col("customer_id"),F.col("email")))

In [0]:
from pyspark.sql.functions import col
display(customers_df.select(col("customer_id"),col("email").alias("customer_email"),col("updated"),col("profile")))

In [0]:
display(customers_df.drop("updated","profile"))

In [0]:
from pyspark.sql.functions import split
display(customers_df.withColumn("domain",split(col("email"),"@")[1])
        .withColumnRenamed("email","customer_email")
        .drop("profile","updated"))
#display(customers_df)

In [0]:
new_customers_df = (customers_df.withColumn("domain",split(col("email"),"@")[1])
        .withColumnRenamed("email","customer_email")
        .drop("profile","updated"))
display(new_customers_df)

In [0]:
orders_df = spark.table("orders")
display(orders_df.describe())

In [0]:
display(orders_df.summary())

In [0]:
display(customers_df.describe())

In [0]:

display(customers_df.dropna())

In [0]:

display(customers_df.na.drop())

In [0]:
display(customers_df.na.fill({"email":"unknown@example.com"}))

In [0]:
joined_df = customers_df.join(orders_df,"customer_id","left").filter("order_id IS NULL")
display(joined_df)

In [0]:
joined_df = customers_df.join(orders_df,"customer_id","left_anti")
display(joined_df)

In [0]:
joined_df = customers_df.join(orders_df,"customer_id","left_semi")
display(joined_df)

In [0]:
from pyspark.sql.functions import explode,broadcast

books_df = spark.table("books")

exploded_orders_df = orders_df.withColumn("book",explode("books")) \
                            .select("*","book.book_id","book.subtotal") \
                            .drop("book","books")
order_details_df = exploded_orders_df.join(broadcast(books_df),"book_id","inner")
display(order_details_df)

In [0]:
order_details_df.write.mode("overwrite").saveAsTable('order_details')
display(spark.table("order_details"))

In [0]:
%sql
select count(*) from order_details